In [68]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

In [69]:
df = pd.read_csv("/content/customer_churn.csv")

In [70]:
print(df.shape)

(1000000, 32)


In [71]:
print(df.head())

      customer_id                 signup_date  age  gender  annual_income  \
0  CUST0000000001  2022-12-12 12:53:58.199463   43  Female       55085.25   
1  CUST0000000002  2022-01-13 12:53:58.199973   18    Male       60786.11   
2  CUST0000000003  2023-09-04 12:53:58.199985   38  Female       73184.32   
3  CUST0000000004  2022-06-27 12:53:58.199992   44    Male       40923.78   
4  CUST0000000005  2022-12-08 12:53:58.199999   45  Female       36400.94   

     education marital_status  dependents  tenure  contract  ...  \
0      college        married           1       2  two_year  ...   
1       master        married           1      22  one_year  ...   
2  high_school        widowed           0       3  two_year  ...   
3  high_school        married           1       6  two_year  ...   
4     bachelor         single           0       9  two_year  ...   

  has_streaming_tv has_streaming_movies  customer_satisfaction  \
0                1                    1                    9.0

In [72]:
print(df.isnull().sum())

customer_id                        0
signup_date                        0
age                                0
gender                             0
annual_income                  29959
education                          0
marital_status                     0
dependents                         0
tenure                             0
contract                           0
payment_method                     0
paperless_billing                  0
senior_citizen                     0
monthlycharges                     0
totalcharges                       0
num_services                       0
has_phone_service                  0
has_internet_service               0
has_online_security                0
has_online_backup                  0
has_device_protection              0
has_tech_support                   0
has_streaming_tv                   0
has_streaming_movies               0
customer_satisfaction          19921
num_complaints                 29906
num_service_calls                  0
l

In [73]:
df['annual_income']= df['annual_income'].fillna(df['annual_income'].median())
df['customer_satisfaction']= df['customer_satisfaction'].fillna(df['customer_satisfaction'].median())
df['num_complaints']= df['num_complaints'].fillna(df['num_complaints'].median())
df['avg_monthly_gb']= df['avg_monthly_gb'].fillna(df['avg_monthly_gb'].median())
df['credit_score']= df['credit_score'].fillna(df['credit_score'].median())

In [74]:
df = df.drop_duplicates()

In [75]:
df = df.drop(columns = ['customer_id', 'signup_date'])

In [76]:
df= pd.get_dummies(df, drop_first=True)

In [77]:
print(df.select_dtypes(include='object').columns)

Index([], dtype='object')


In [78]:
x= df.drop(columns=['churn'])
y= df['churn']

In [79]:
print(x.head())
print('-*-'*100)
print(y.head())

   age  annual_income  dependents  tenure  senior_citizen  monthlycharges  \
0   43       55085.25           1       2               0           67.20   
1   18       60786.11           1      22               0           71.54   
2   38       73184.32           0       3               0          112.20   
3   44       40923.78           1       6               0          107.49   
4   45       36400.94           0       9               0          110.05   

   totalcharges  num_services  has_phone_service  has_internet_service  ...  \
0        144.39             1                  1                     1  ...   
1       1602.22             2                  0                     1  ...   
2        328.81             4                  1                     1  ...   
3        643.45             3                  1                     1  ...   
4        648.79             3                  0                     1  ...   

   education_phd  marital_status_married  marital_status_singl

In [80]:
x_train,x_test,y_train,y_test=train_test_split(x,y, test_size=0.2, stratify=y, random_state=42)

In [99]:
model= LogisticRegression(max_iter=1000, class_weight = 'balanced')

In [100]:
model.fit(x_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(class_weight='balanced', max_iter=1000)

In [101]:
y_pred =model.predict(x_test)

In [102]:
y_prob = model.predict_proba(x_test)[:,1]

In [103]:
print("ROC AUC:",roc_auc_score(y_test, y_prob))
print("\n Classification Report:\n")
print(classification_report(y_test, y_pred))

ROC AUC: 0.6327170342992697

 Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.62      0.74    180155
           1       0.14      0.57      0.23     19845

    accuracy                           0.61    200000
   macro avg       0.54      0.60      0.48    200000
weighted avg       0.85      0.61      0.69    200000



In [104]:
df['Predicted_Churn']= model.predict(x)
df.to_csv('Churn Dashboard data.csv', index = False)